# House Rent Prediction with KNN

## Step 1 - Load and inspect the dataset

Load the dataset with Pandas in the cell below and display its first five rows.

In [ ]:
import pandas as pd

data_path = "../data/raw/House_Rent_Dataset.csv"
df = pd.read_csv(data_path)

df.head()

## Step 2 - Understand the dataset structure

Check the dataset size, column data types, and missing values.

In [ ]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.info()

## Step 3 - Summarize the numerical columns

Review the center, spread, and range of the numerical variables.

In [ ]:
df[["BHK", "Rent", "Size", "Bathroom"]].describe().round(2)

## Step 4 - Explore the target variable

Visualize `Rent` to check its distribution, skewness, and extreme values.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(df["Rent"], bins=40, edgecolor="black")
plt.axvline(df["Rent"].median(), color="red", linestyle="--", label="Median")
plt.title("Distribution of Monthly Rent")
plt.xlabel("Rent")
plt.ylabel("Number of Listings")
plt.legend()
plt.tight_layout()
plt.savefig("../images/rent_distribution.png", dpi=150)
plt.show()

## Step 5 - Select the initial features and target

Use three numerical features for the first model and keep `Rent` as the prediction target.

In [ ]:
features = ["BHK", "Size", "Bathroom"]
target = "Rent"

X = df[features]
y = df[target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

## Step 6 - Split the data

Use 70% of the data for training, 15% for validation, and 15% for final testing.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

## Step 7 - Scale the features

Fit the scaler only on the training data, then apply the same transformation to validation and test data.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

pd.DataFrame(X_train_scaled, columns=features).describe().round(2)

## Step 8 - Build a mean baseline

Predict the training-set mean for every validation sample and measure the mean absolute error.

In [ ]:
baseline_prediction = y_train.mean()
baseline_mae = (y_val - baseline_prediction).abs().mean()

print(f"Training rent mean: {baseline_prediction:,.0f}")
print(f"Baseline validation MAE: {baseline_mae:,.0f}")

## Step 9 - Train the first KNN model

Implement KNN from scratch with Euclidean distance, start with `k=5`, and compare it with the mean baseline.

In [ ]:
import numpy as np

def knn_predict(X_train, y_train, X_query, k):
    predictions = []
    train_targets = y_train.to_numpy()

    for query in X_query:
        distances = np.linalg.norm(X_train - query, axis=1)
        nearest_indices = np.argsort(distances)[:k]
        predictions.append(train_targets[nearest_indices].mean())

    return np.array(predictions)

val_predictions = knn_predict(X_train_scaled, y_train, X_val_scaled, k=5)
knn_val_mae = np.abs(y_val.to_numpy() - val_predictions).mean()

print(f"KNN validation MAE (k=5): {knn_val_mae:,.0f}")
print(f"Improvement over baseline: {baseline_mae - knn_val_mae:,.0f}")

## Step 10 - Tune the number of neighbors

Compare validation MAE for `k` values from 1 to 30 and select the best value.

In [ ]:
k_values = range(1, 31)
distance_matrix = np.linalg.norm(
    X_val_scaled[:, None, :] - X_train_scaled[None, :, :], axis=2
)
nearest_indices = np.argsort(distance_matrix, axis=1)[:, :max(k_values)]
nearest_rents = y_train.to_numpy()[nearest_indices]
cumulative_predictions = np.cumsum(nearest_rents, axis=1) / np.arange(1, max(k_values) + 1)
validation_mae = [
    np.abs(y_val.to_numpy() - cumulative_predictions[:, k - 1]).mean()
    for k in k_values
]

best_k = k_values[validation_mae.index(min(validation_mae))]
print(f"Best k: {best_k}")
print(f"Best validation MAE: {min(validation_mae):,.0f}")

plt.figure(figsize=(8, 4))
plt.plot(k_values, validation_mae, marker="o")
plt.axvline(best_k, color="red", linestyle="--", label=f"Best k = {best_k}")
plt.title("Validation MAE for Different k Values")
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Validation MAE")
plt.legend()
plt.tight_layout()
plt.savefig("../images/k_vs_validation_mae.png", dpi=150)
plt.show()

## Step 11 - Evaluate the final model

Combine the training and validation data, refit the scaler, and evaluate the selected `k` once on the test set.

In [ ]:
X_final = pd.concat([X_train, X_val])
y_final = pd.concat([y_train, y_val])

final_scaler = StandardScaler()
X_final_scaled = final_scaler.fit_transform(X_final)
X_test_final_scaled = final_scaler.transform(X_test)

test_predictions = knn_predict(
    X_final_scaled, y_final, X_test_final_scaled, best_k
)
test_mae = np.abs(y_test.to_numpy() - test_predictions).mean()

print(f"Final k: {best_k}")
print(f"Test MAE: {test_mae:,.0f}")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_predictions, alpha=0.5)
limits = [
    min(y_test.min(), test_predictions.min()),
    max(y_test.max(), test_predictions.max()),
]
plt.plot(limits, limits, "r--", label="Perfect prediction")
plt.xlabel("Actual Rent")
plt.ylabel("Predicted Rent")
plt.title("Actual vs Predicted Rent")
plt.legend()
plt.tight_layout()
plt.savefig("../images/actual_vs_predicted_rent.png", dpi=150)
plt.show()

## Step 12 - Analyze prediction errors

Compare actual and predicted rents, calculate the absolute error for each test sample, and inspect the largest errors.

In [ ]:
error_analysis = X_test.copy()
error_analysis["Actual Rent"] = y_test
error_analysis["Predicted Rent"] = test_predictions
error_analysis["Absolute Error"] = np.abs(
    error_analysis["Actual Rent"] - error_analysis["Predicted Rent"]
)

print(f"Median absolute error: {error_analysis['Absolute Error'].median():,.0f}")
error_analysis.sort_values("Absolute Error", ascending=False).head(10).round(0)

## Step 13 - Summarize the results

Compare the final KNN model with a mean baseline on the untouched test set.

In [ ]:
final_baseline_prediction = y_final.mean()
baseline_test_mae = np.abs(y_test - final_baseline_prediction).mean()
improvement = 100 * (baseline_test_mae - test_mae) / baseline_test_mae

results = pd.DataFrame({
    "Model": ["Mean baseline", f"KNN (k={best_k})"],
    "Test MAE": [baseline_test_mae, test_mae],
})

print(f"KNN improvement over baseline: {improvement:.1f}%")
results.round(0)

## Conclusion

- KNN regression can estimate rent by averaging the rents of similar houses.
- Feature scaling is essential because KNN is based on distance.
- The validation set was used to select `k`; the test set was used only once for final evaluation.
- The model uses only `BHK`, `Size`, and `Bathroom`, so it cannot capture important location and property-condition effects.
- Future work could encode features such as `City`, `Area Type`, and `Furnishing Status` and compare the result without using the test set for tuning.